# Harvest DataCite dataset records up to September 2025

## Import

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sindex.sources.datacite.jobs import (
    harvest_datacite_datasets_for_date_range_to_ndjson,
    batch_slim_datacite_record_to_ndjson,
    batch_slim_datacite_record_to_ndjson_fast
    )
from sindex.utils.datasets import create_datasets_db_from_ndjson
from pathlib import Path
import os
import duckdb

## Query datacite and write to ndjson files

In [ ]:
# Test parameters
#start_date_str = "2014-12-28"
#end_date_str = "2015-01-10"
#out_dir = Path("output/datacite/harvest_date_range_test")

# Actual
start_date_str = "2011-03-08"
end_date_str = "2025-07-29" #update this date if code fails to the one that was running
out_dir = Path("C:/Users/BPatel/Documents/batch-data/datacite-raw")

out_dir.mkdir(parents=True, exist_ok=True)

n = harvest_datacite_datasets_for_date_range_to_ndjson(
    start_date_str=start_date_str,
    end_date_str=end_date_str,
    window_days=7,
    page_size=1000,
    save_folder=str(out_dir),
    skip_empty_files=True,
    polite_sleep_seconds=0.5,
)

print(f"Done. Wrote {n:,} records across window files in {out_dir.resolve()}")

Fetching records 2025-07-23 → 2025-07-29 (window_days=7, page_size=1000, detail=True)
  Saved 302574 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2025-07-23-2025-07-29.ndjson
Fetching records 2025-07-16 → 2025-07-22 (window_days=7, page_size=1000, detail=True)
  Saved 433070 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2025-07-16-2025-07-22.ndjson
Fetching records 2025-07-09 → 2025-07-15 (window_days=7, page_size=1000, detail=True)
  Saved 634424 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2025-07-09-2025-07-15.ndjson
Fetching records 2025-07-02 → 2025-07-08 (window_days=7, page_size=1000, detail=True)
  Saved 1103051 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2025-07-02-2025-07-08.ndjson
Fetching records 2025-06-25 → 2025-07-01 (window_days=7, page_size=1000, detail=True)
  Saved 818752 records → C:\Users\BPatel\Documents\batch-data\datacite-raw\datacite-2025-06-25-2025-07-01.ndjson
Fetc

---------------
---------------
Check results on DataCite commons: https://commons.datacite.org/doi.org?query=%28types.resourceTypeGeneral%3ADataset%29+AND+%28created%3A%5B2025-09-24+TO+2025-09-30%5D%29&registration-agency=datacite

## Check number of records

In [7]:
# In one file
def count_ndjson_lines(file_path):
    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  
                count += 1
    return count
file_path = os.path.join(out_dir, "datacite-2025-07-30-2025-08-05.ndjson")
count = count_ndjson_lines(file_path)
print(f"Total lines: {count}")

Total lines: 1394000


In [ ]:
# Across all files

## Write batch slim records

In [9]:
src_folder = Path(r"I:\pipeline-data\raw-records\datacite-records")
dst_folder = Path(r"I:\pipeline-data\records\slim-records\datacite-slim-records")
summary = batch_slim_datacite_record_to_ndjson_fast(
    src_folder=str(src_folder),
    dst_folder=str(dst_folder),
    overwrite=True,          # overwrite for repeatable tests
    accept_gz=True,
    one_line_progress=True
)

# Summary
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\nCompleted.")

Running DataCite batch slimming test
  Input folder : \\192.168.20.168\AILarge\pipeline-data\raw-records\datacite-records
  Output folder: \\192.168.20.168\AILarge\pipeline-data\records\slim-records\datacite-slim-records
---
Processing 771 files using 40 cores...
[771/771] files completed
Done. files=771 kept=49,009,522 bad=0 time=5609.2s rateâ‰ˆ8,737/rec-per-sec

Summary:
  files_seen: 771
  records_read: 49009522
  records_kept: 49009522
  records_bad_json: 0
  output_dir: \\192.168.20.168\AILarge\pipeline-data\records\slim-records\datacite-slim-records
  elapsed_sec: 5609.18
  rate_rec_per_sec: 8737

Completed.


## Slim records to duckdb

### Test

In [17]:
slim_folder = r"C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks\input\demo_datasets_slim_metadata_ndjson"
db_path = r"C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks-batch\output\datacite\test_datacite_slim_records.duckdb"
create_datasets_db_from_ndjson(slim_folder, db_path)

1/3: Extracting data from 2 files...
2/3: Creating index for fast lookups (this may take a moment)...
3/3: Saving to disk...

Success! Persistent DB created at: C:\Users\Admin\Documents\GitHub\sindex-pipeline\notebooks-batch\output\datacite\test_datacite_slim_records.duckdb
Total indexed datasets: 3


In [18]:
con = duckdb.connect(db_path)
row_count = con.execute("SELECT count() FROM my_datasets").fetchone()[0]
print(f"Total datasets in table: {row_count}")
display(con.execute("SELECT * FROM my_datasets LIMIT 5").df())
con.close()

Total datasets in table: 3


,dataset_id,id_type,publication_date,created_date,publication_year
0,10.13026/kpb9-mt58,doi,2024-10-10T21:27:19+00:00,None,2024
1,10.60775/fairhub.2,doi,2024-10-28T17:30:18+00:00,None,2024
2,EMD-24511,emdb_id,2021-07-22T00:00:00,None,2021


### Actual

In [3]:
slim_folder = r"I:\pipeline-data\records\slim-records\datacite-slim-records"
db_path = r"I:\pipeline-data\records\slim-records\datacite-slim-records.duckdb"

In [5]:
create_datasets_db_from_ndjson(slim_folder, db_path)

Extracting data from 769 files


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Creating index for fast lookups (this may take a moment)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saving to disk

Success! Persistent DB created at: I:\pipeline-data\records\slim-records\datacite-slim-records.duckdb
Total indexed datasets: 49,009,522


In [6]:
con = duckdb.connect(db_path)
row_count = con.execute("SELECT count() FROM my_datasets").fetchone()[0]
print(f"Total datasets in table: {row_count:,}")
display(con.execute("SELECT * FROM my_datasets LIMIT 5").df())
con.close()

Total datasets in table: 49,009,522


,dataset_id,id_type,publication_date,created_date,publication_year
0,10.5284/1000389,doi,2011-01-01T00:00:00,2011-03-09T17:02:45+00:00,2011
1,10.5284/1000140,doi,2011-01-01T00:00:00,2011-03-10T13:03:09+00:00,2011
2,10.5284/1000146,doi,2011-01-01T00:00:00,2011-03-10T14:56:29+00:00,2011
3,10.5284/1000144,doi,2011-01-01T00:00:00,2011-03-10T15:11:32+00:00,2011
4,10.5284/1000181,doi,2011-01-01T00:00:00,2011-03-10T15:22:56+00:00,2011


In [7]:
# Export to CSV
con = duckdb.connect(db_path)

copy_query = """
COPY (
    SELECT dataset_id 
    FROM my_datasets 
    WHERE dataset_id IS NOT NULL AND dataset_id != ''
) TO 'exported_dataset_ids.csv' (HEADER);
"""

# Execute the command
con.execute(copy_query)
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))